# QLoRA Summarization — Colab Training Pipeline

In [ ]:
GITHUB_REPO    = "https://github.com/ducnd58233/language-engineer.git"
GITHUB_TOKEN   = ""
HF_TOKEN       = ""
HF_REPO_ID     = "ducnd58233/qwen2.5-3b-qlora-summarization"
MAX_STEPS      = 20
N_EVAL_SAMPLES = 100

## 1. Setup

In [2]:
import os
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or GITHUB_TOKEN
except Exception:
    pass

repo_url = GITHUB_REPO.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else GITHUB_REPO

if not os.path.exists("language-engineer"):
    subprocess.run(["git", "clone", repo_url, "language-engineer"], check=True)
else:
    subprocess.run(["git", "-C", "language-engineer", "pull"], check=True)

In [3]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q peft trl transformers accelerate bitsandbytes datasets \
    rouge-score sacrebleu bert-score python-dotenv pyyaml tqdm

In [4]:
%cd language-engineer

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or HF_TOKEN
except Exception:
    pass

with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")
    f.write(f"HF_REPO_ID={HF_REPO_ID}\n")

/content/language-engineer


## 2. (Optional) Mount Google Drive

Uncomment to persist checkpoints and datasets across Colab sessions.
Without this, all data is lost when the runtime disconnects.

In [5]:
# from google.colab import drive
# drive.mount("/content/drive")
#
# os.makedirs("/content/drive/MyDrive/lang-eng/runs", exist_ok=True)
# os.makedirs("/content/drive/MyDrive/lang-eng/datasets", exist_ok=True)
# if not os.path.exists("runs"):
#     os.symlink("/content/drive/MyDrive/lang-eng/runs", "runs")
# if not os.path.exists("datasets"):
#     os.symlink("/content/drive/MyDrive/lang-eng/datasets", "datasets")

## 3. Prepare Datasets

In [ ]:
!python scripts/prepare_datasets.py --limit-rows 5000

In [7]:
!python scripts/process_datasets.py

Datasets found: ['cnn', 'xsum']
Generating train split: 287113 examples [00:56, 5105.62 examples/s] 
Generating train split: 204017 examples [00:03, 65392.76 examples/s]
Generating train split: 13368 examples [00:00, 45387.69 examples/s]
Generating train split: 11327 examples [00:00, 64807.49 examples/s]
Generating train split: 11490 examples [00:00, 20478.93 examples/s]
Generating train split: 11333 examples [00:02, 4888.11 examples/s]
[leakage] removed 10 rows from validation found in train
[leakage] removed 9 rows from test found in train

Split            Raw     Hard      IQR    Dedup    Final
----------------------------------------------------------------
hard filter: 100% 491130/491130 [00:34<00:00, 14396.54 examples/s]
train        491,130  -10,500  -28,576  -3,050  449,004
format prompts: 100% 449004/449004 [02:07<00:00, 3517.94 examples/s]
Creating parquet from Arrow format: 100% 450/450 [01:02<00:00,  7.20ba/s]
Saved train -> /content/language-engineer/datasets/processed/tr

## 4. Train

In [8]:
!python scripts/train.py --max-steps {MAX_STEPS}

Loading tokenizer: Qwen/Qwen2.5-3B-Instruct
Loading model: Qwen/Qwen2.5-3B-Instruct (4-bit NF4)
Loading weights: 100% 434/434 [00:30<00:00, 14.39it/s, Materializing param=model.norm.weight]                               
trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193
Loading datasets...
Generating train split: 449004 examples [01:45, 4242.27 examples/s] 
Generating train split: 22934 examples [00:01, 11804.62 examples/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Adding EOS to train dataset: 100% 449004/449004 [02:06<00:00, 3560.82 examples/s]
Tokenizing train dataset: 100% 449004/449004 [44:39<00:00, 167.57 examples/s] 
Truncating train dataset:  71% 321000/449004 [01:03<00:25, 5022.56 examples/s] 
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/datasets/arrow_dataset.py", line 3681, in _map_single
    writer.write_table(batch)
  File "/usr/local/lib/python3.12/dist-packages/dat

## 5. Evaluate

Compares base model (zero-shot) vs. fine-tuned adapter on the test set.
Results saved to `results/eval_results.json`.

In [ ]:
!python scripts/evaluate.py --n-samples {N_EVAL_SAMPLES}

Loading test data from /content/language-engineer/datasets/processed/test/data.parquet
Generating train split: 21065 examples [00:01, 14573.34 examples/s]
Loading tokenizer: Qwen/Qwen2.5-3B-Instruct

--- Evaluating base model (zero-shot) ---
Loading weights: 100% 434/434 [00:25<00:00, 17.29it/s, Materializing param=model.norm.weight]                               m=model.layers.16.self_attn.q_proj.weight]

[base] Generating 100 summaries...
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


: 

## 6. Upload to HuggingFace Hub

In [ ]:
!python scripts/hub.py upload \
    --adapter runs/Qwen2.5-3B-Instruct_qlora/final \
    --repo {HF_REPO_ID}

Traceback (most recent call last):
  File "/content/language-engineer/scripts/hub.py", line 44, in <module>
    upload(args.adapter, args.repo, args.private)
  File "/content/language-engineer/scripts/hub.py", line 16, in upload
    api.upload_folder(folder_path=adapter_path, repo_id=repo_id, repo_type="model")
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 89, in _inner_fn
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py", line 1857, in _inner
    return fn(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py", line 5193, in upload_folder
    add_operations = self._prepare_upload_folder_additions(
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py", line 10148, in _prepare_upload_folder_additio

## 7. (Optional) Download from HuggingFace Hub

Pull a previously uploaded adapter back to local disk (e.g. to resume eval on a new runtime).

In [ ]:
!python scripts/hub.py download \
    --repo {HF_REPO_ID} \
    --output runs/downloaded

Fetching 1 files: 100% 1/1 [00:00<00:00,  3.49it/s]
Download complete: : 1.52kB [00:00, 5.31kB/s]              Downloaded ducnd58233/qwen2.5-3b-qlora-summarization -> runs/downloaded
Download complete: : 1.52kB [00:00, 4.84kB/s]
